In [50]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.linear_model import Lasso, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

import statsmodels.api as sm

import os, json, re, time
from pathlib import Path


In [51]:
# ── Load data ──

topics_raw = pd.read_csv('../data3/topics.csv')
macro_raw  = pd.read_csv('../data3/macro.csv')
articles   = pd.read_parquet('../data3/articles.pq')

# Parse dates + set monthly index (matches HW3p1 conventions)
topics_raw['date'] = pd.to_datetime(topics_raw['date']).dt.to_period('M').dt.to_timestamp()
macro_raw['date']  = pd.to_datetime(macro_raw['date']).dt.to_period('M').dt.to_timestamp()
articles['display_date'] = pd.to_datetime(articles['display_date'])

topics = topics_raw.set_index('date').sort_index()
macro  = macro_raw.set_index('date').sort_index()

print('topics:', topics.shape)
print('macro :', macro.shape)
print('articles:', articles.shape)


topics: (402, 180)
macro : (444, 53)
articles: (10200, 3)


In [52]:
# ── Restrict both datasets to overlapping dates ──
idx = topics.index.intersection(macro.index)
topics = topics.loc[idx]
macro  = macro.loc[idx]

print('Overlapping months:', len(idx))
print('Date range:', idx.min().date(), 'to', idx.max().date())


Overlapping months: 402
Date range: 1984-01-01 to 2017-06-01


---
## Q2(a) — Identify the Q1-selected topics, then generate them from headlines

We first reproduce the **Q1-style** selection (lasso sweep → choose α with 5 non-zeros → OLS R²) using `topics.csv`. Then we ask an LLM to score each headline against the union of those selected topics.

---


In [53]:
# Outcomes: mret, vol, indpro, indprol1, plus any *_vol columns (matches PS text)
outcomes = ['mret', 'vol', 'indpro', 'indprol1'] + [c for c in macro.columns if c.endswith('_vol') and c != 'vol']
outcomes = [c for c in outcomes if c in macro.columns]
outcomes


['mret',
 'vol',
 'indpro',
 'indprol1',
 'Agric_vol',
 'Food_vol',
 'Soda_vol',
 'Beer_vol',
 'Smoke_vol',
 'Toys_vol',
 'Fun_vol',
 'Books_vol',
 'Hshld_vol',
 'Clths_vol',
 'Hlth_vol',
 'MedEq_vol',
 'Drugs_vol',
 'Chems_vol',
 'Rubbr_vol',
 'Txtls_vol',
 'BldMt_vol',
 'Cnstr_vol',
 'Steel_vol',
 'FabPr_vol',
 'Mach_vol',
 'ElcEq_vol',
 'Autos_vol',
 'Aero_vol',
 'Ships_vol',
 'Guns_vol',
 'Gold_vol',
 'Mines_vol',
 'Coal_vol',
 'Oil_vol',
 'Util_vol',
 'Telcm_vol',
 'PerSv_vol',
 'BusSv_vol',
 'Hardw_vol',
 'Softw_vol',
 'Chips_vol',
 'LabEq_vol',
 'Paper_vol',
 'Boxes_vol',
 'Trans_vol',
 'Whlsl_vol',
 'Rtail_vol',
 'Meals_vol',
 'Banks_vol',
 'Insur_vol',
 'RlEst_vol',
 'Fin_vol',
 'Other_vol']

In [54]:
def lasso_pick_alpha_k_nonzero(X_scaled, y, alphas=None, k=5):
    # mirrors HW3p1: sweep alphas, find first alpha with exactly k non-zero coefficients
    if alphas is None:
        alphas = np.logspace(-6, 0, 300)
    n_nonzero = []
    coef_paths = []
    for a in alphas:
        lasso = Lasso(alpha=a, max_iter=50_000)
        lasso.fit(X_scaled, y)
        n_nonzero.append(np.sum(lasso.coef_ != 0))
        coef_paths.append(lasso.coef_.copy())
    n_nonzero = np.array(n_nonzero)
    coef_paths = np.array(coef_paths)

    mask_k = n_nonzero == k
    if mask_k.any():
        alpha_k = alphas[mask_k][0]  # smallest alpha with exactly k
    else:
        # if exact k never occurs, pick closest
        alpha_k = alphas[np.argmin(np.abs(n_nonzero - k))]
    return alpha_k, n_nonzero, coef_paths, alphas

def ols_r2(X, y):
    X_ = sm.add_constant(X)
    model = sm.OLS(y, X_).fit()
    return model.rsquared, model


In [55]:
# ── Use selected topics from Q1 (hard-coded from your results) ──

selected_topics = {

    # Q1(a)
    "mret": [
        "Problems",
        "Federal Reserve",
        "Recession",
        "Bear/bull market",
        "Options/VIX"
    ],

    # Q1(b)
    "vol": ["Small business", "Problems", "Recession", "Investment banking", "Options/VIX"],
    "indpro": ["Recession", "Space program", "Clintons", "Southeast Asia", "Oil market"],
    "indprol1": ["Russia", "Health insurance", "Recession", "Space program", "Oil market"],

    "Food_vol": ["Drexel", "Investment banking", "Southeast Asia", "Mortgages", "Retail"],
    "Soda_vol": ["Japan", "International exchanges", "China", "Trading activity", "Lawsuits"],
    "Beer_vol": ["China", "Futures/indices", "Private equity/hedge funds", "Retail", "Revenue growth"],
    "Smoke_vol": ["News conference", "Economic growth", "Financial crisis", "Tobacco", "Exchanges/composites"],
    "Fun_vol": ["Competition", "European sovereign debt", "Financial crisis", "Marketing", "Exchanges/composites"],
    "Books_vol": ["Police/crime", "Financial crisis", "International exchanges", "Exchanges/composites", "European politics"],
    "Hshld_vol": ["Activists", "Environment ", "Control stakes", "Recession", "Credit cards"],
    "Clths_vol": ["Cable", "C-suite", "SEC", "Phone companies", "European politics"],
    "Hlth_vol": ["Record high", "Mining", "Publishing", "Latin America", "Product prices"],
    "MedEq_vol": ["Earnings losses", "China", "Mid-level executives", "Financial reports", "Private equity/hedge funds"],
    "Drugs_vol": ["European sovereign debt", "Oil drilling", "Clintons", "Mortgages", "Foods/consumer goods"],
    "Chems_vol": ["Internet", "M&A", "Savings & loans", "Russia", "Bankruptcy"],
    "Rubbr_vol": ["Fast food", "Bankruptcy", "Arts", "Clintons", "Mortgages"],
    "Txtls_vol": ["Mid-size cities", "California", "Futures/indices", "Acquired investment banks", "Microchips"],
    "BldMt_vol": ["Mobile devices", "European sovereign debt", "Currencies/metals", "Oil market", "European politics"],
    "Cnstr_vol": ["Executive pay", "SEC", "California", "Clintons", "Futures/indices"],
    "Steel_vol": ["Problems", "China", "Futures/indices", "Bush/Obama/Trump", "Private equity/hedge funds"],
    "ElcEq_vol": ["Mid-size cities", "International exchanges", "California", "Futures/indices", "National security"],
    "Autos_vol": ["Small business", "Treasury bonds", "Latin America", "Marketing", "Automotive"],
    "Guns_vol": ["Research", "Schools", "Humor/language", "Options/VIX", "European politics"],
    "Mines_vol": ["Mining", "China", "Futures/indices", "Mortgages", "Private equity/hedge funds"],
    "Coal_vol": ["Changes", "Treasury bonds", "Immigration", "China", "Commodities"],
    "Oil_vol": ["M&A", "Problems", "Private/public sector", "Financial crisis", "Oil market"],
    "Util_vol": ["Treasury bonds", "Challenges", "Commodities", "National security", "Revenue growth"],
    "Telcm_vol": ["Mobile devices", "Programs/initiatives", "China", "Humor/language", "Futures/indices"],
    "PerSv_vol": ["Environment ", "Venture capital", "Financial crisis", "Foods/consumer goods", "Trading activity"],
    "BusSv_vol": ["Electronics", "US defense", "Health insurance", "SEC", "Accounting"],
    "Hardw_vol": ["Phone companies", "Tobacco", "Oil market", "Microchips", "NASD"],
    "Softw_vol": ["Key role", "Justice Department", "China", "Futures/indices", "Acquired investment banks"],
    "Chips_vol": ["Small caps", "Mining", "Phone companies", "Bank loans", "NASD"],
    "LabEq_vol": ["Record high", "Bond yields", "Health insurance", "SEC", "Reagan"],
    "Paper_vol": ["M&A", "Exchanges/composites", "Bush/Obama/Trump", "Microchips", "Iraq"],
    "Boxes_vol": ["Company spokesperson", "Marketing", "International exchanges", "Utilities", "Trading activity"],
    "Trans_vol": ["M&A", "Control stakes", "Iraq", "Buffett", "Lawsuits"],
    "Rtail_vol": ["Electronics", "Broadcasting", "Schools", "Germany", "Oil market"],
    "Meals_vol": ["Company spokesperson", "Pharma", "Marketing", "Systems", "Reagan"],
    "Banks_vol": ["Financial crisis", "SEC", "Accounting", "Options/VIX", "NASD"],
    "Insur_vol": ["Credit ratings", "Company spokesperson", "Financial crisis", "International exchanges", "Bank loans"],
    "RlEst_vol": ["Publishing", "Automotive", "International exchanges", "Mortgages", "National security"],
    "Fin_vol": ["Canada/South Africa", "Mortgages", "Subsidiaries", "Taxes", "Private equity/hedge funds"],
    "Other_vol": ["Treasury bonds", "Japan", "International exchanges", "Futures/indices", "Elections"],
}

topics_union = sorted(set(sum(selected_topics.values(), [])))

print("Total unique topics to score:", len(topics_union))

Total unique topics to score: 102


---
### LLM scoring setup
We score each headline against the union topics list. To keep prompts short, we chunk the topic list and combine responses.

---


In [56]:
# ── OpenAI client + key loading ──
def load_api_key():
    k = os.environ.get('OPENAI_API_KEY', '').strip()
    if k:
        return k
    key_path = Path('../data3/key.txt')
    if key_path.exists():
        for line in key_path.read_text().splitlines():
            line = line.strip()
            if line and not line.startswith('#'):
                return line
    raise RuntimeError('No API key found. Set OPENAI_API_KEY or put it in key.txt')

from openai import OpenAI
client = OpenAI(api_key=load_api_key())

MODEL = os.environ.get('OPENAI_MODEL', 'gpt-4.1-nano')


In [57]:
import re, json, time
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── Prompt builder ──
def build_prompt(headline, topics_list, persona=None, ignore_future=False):
    persona_intro = ""
    if persona == "bull":
        persona_intro = "You are an overly optimistic investor who sees opportunity in every situation.\n"
    elif persona == "bear":
        persona_intro = "You are a deeply skeptical investor who sees risk and danger in market developments.\n"

    future_clause = ""
    if ignore_future:
        future_clause = (
            "IMPORTANT: Ignore any knowledge of events after the headline date. "
            "Only infer what a reader at the time could reasonably infer.\n\n"
        )

    topics_str = ", ".join([f'"{t}"' for t in topics_list])

    system_msg = (
        persona_intro
        + "You extract economic and financial topics/risk factors from news headlines."
    ).strip()

    user_msg = f"""{future_clause}Given the headline below, mark whether each topic is relevant (1) or not relevant (0).

Topics: [{topics_str}]

Headline: "{headline}"

Return ONLY valid JSON mapping each topic name to 0 or 1."""

    return system_msg, user_msg


# ── LLM call helper ──
def call_llm_json(system_msg, user_msg, temperature=0.3, max_retries=4):
    for attempt in range(max_retries):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                temperature=temperature,
                max_tokens=2000,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg},
                ],
            )
            text = (resp.choices[0].message.content or "").strip()
            m = re.search(r"\{[\s\S]*\}", text)
            if not m:
                raise ValueError("No JSON object found")
            obj = json.loads(m.group(0))
            return {k: (1 if str(v).strip() in ["1", "true", "True"] else 0) for k, v in obj.items()}
        except Exception as e:
            if attempt == max_retries - 1:
                raise RuntimeError(f"LLM call failed: {e}")
            time.sleep(min(5.0, (2 ** attempt) * 0.3))


def score_single_headline(headline, topics_list, temperature, persona, ignore_future):
    """Score one headline against all topics in a single API call."""
    sys_msg, usr_msg = build_prompt(headline, topics_list, persona=persona, ignore_future=ignore_future)
    out = call_llm_json(sys_msg, usr_msg, temperature=temperature)
    return {t: int(out.get(t, 0)) for t in topics_list}

In [58]:
# ── Article-level scoring with caching + parallelism ──
CACHE_DIR = Path('../data3/outputs_q2')
CACHE_DIR.mkdir(parents=True, exist_ok=True)

MAX_WORKERS = 10  # parallel API calls

def score_headlines(topics_union, temperature=0.3, persona=None, ignore_future=False, limit=None):
    slug = f'model={MODEL}__temp={str(temperature).replace(".","p")}__persona={persona or "none"}__ignorefuture={int(ignore_future)}__limit={limit or "all"}'
    cache_path = CACHE_DIR / f'llm_scores_{slug}.parquet'
    if cache_path.exists():
        print(f"  [cache hit] {cache_path.name}")
        return pd.read_parquet(cache_path)

    d = articles[['display_date', 'headline']].copy()
    d['headline'] = d['headline'].astype(str)
    if limit is not None:
        d = d.head(limit).copy()

    headlines = d['headline'].tolist()
    n = len(headlines)
    print(f"  Scoring {n} headlines × {len(topics_union)} topics ({MAX_WORKERS} workers)...")

    # Parallel scoring: 1 API call per headline (all topics at once)
    results = [None] * n
    done = [0]

    def _task(idx, hl):
        return idx, score_single_headline(hl, topics_union, temperature, persona, ignore_future)

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(_task, i, hl): i for i, hl in enumerate(headlines)}
        for fut in as_completed(futures):
            idx, scores = fut.result()
            results[idx] = scores
            done[0] += 1
            if done[0] % 100 == 0 or done[0] == n:
                print(f"    {done[0]}/{n} done")

    # Assemble into DataFrame
    scores_df = pd.DataFrame(results)
    for t in topics_union:
        if t not in scores_df.columns:
            scores_df[t] = 0
    d = pd.concat([d.reset_index(drop=True), scores_df[topics_union]], axis=1)

    d.to_parquet(cache_path, index=False)
    print(f"  Saved to {cache_path.name}")
    return d

In [59]:
def monthly_aggregate(article_scores, topics_union):
    d = article_scores.copy()
    d['date'] = pd.to_datetime(d['display_date']).dt.to_period('M').dt.to_timestamp()
    # mean = share of headlines in month triggering that topic
    monthly = d.groupby('date')[topics_union].mean().reset_index().sort_values('date')
    return monthly

---
### Q2(a) — Baseline generation and R²
We run the same contemporaneous lasso→OLS procedure as in HW3p1, but using **LLM-generated monthly topic features**.

---


In [60]:
# Baseline config
BASE_TEMP = 0.3
ARTICLE_LIMIT = 100  # set e.g. 5000 to debug

scores_base = score_headlines(topics_union, temperature=BASE_TEMP, persona=None, ignore_future=False, limit=ARTICLE_LIMIT)
monthly_base = monthly_aggregate(scores_base, topics_union)

df_base = macro.reset_index().merge(monthly_base, on='date', how='inner').sort_values('date').reset_index(drop=True)
X_cols = topics_union

print('monthly_base:', monthly_base.shape)
print('df_base:', df_base.shape)


  [cache hit] llm_scores_model=gpt-4.1-nano__temp=0p3__persona=none__ignorefuture=0__limit=100.parquet
monthly_base: (4, 103)
df_base: (4, 156)


In [61]:
# ── Contemporaneous evaluation (mirrors HW3p1) ──
q2a_rows = []
for TARGET in outcomes:
    y = df_base.set_index('date')[TARGET].dropna()
    X = df_base.set_index('date').loc[y.index, X_cols]
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    alpha_5, n_nonzero, coef_paths, alphas = lasso_pick_alpha_k_nonzero(X_scaled, y.values, k=5)
    lasso_5 = Lasso(alpha=alpha_5, max_iter=50_000).fit(X_scaled, y.values)
    selected = X.columns[lasso_5.coef_ != 0].tolist()

    r2, model = ols_r2(X[selected].values, y.values)

    q2a_rows.append({
        'outcome': TARGET,
        'alpha': alpha_5,
        'selected': '; '.join(selected),
        'ols_r2': r2,
        'n_obs': len(y)
    })
    print(f'{TARGET:10s} | OLS R2={r2:.4f} | selected={selected}')

q2a = pd.DataFrame(q2a_rows).sort_values('ols_r2', ascending=False)
q2a.to_csv(CACHE_DIR / 'q2a_baseline_contemporaneous.csv', index=False)
q2a


mret       | OLS R2=1.0000 | selected=['California', 'Commodities', 'Currencies/metals', 'M&A', 'Treasury bonds']
vol        | OLS R2=1.0000 | selected=['Bond yields', 'C-suite', 'Commodities', 'Exchanges/composites', 'Reagan', 'Research']
indpro     | OLS R2=1.0000 | selected=['Bush/Obama/Trump', 'Commodities', 'Reagan', 'Research', 'Retail']
indprol1   | OLS R2=1.0000 | selected=['Bond yields', 'Currencies/metals', 'Electronics', 'Japan', 'M&A']
Agric_vol  | OLS R2=1.0000 | selected=['Accounting', 'Bankruptcy', 'C-suite', 'Earnings losses', 'Futures/indices']
Food_vol   | OLS R2=1.0000 | selected=['Germany', 'Japan', 'Justice Department', 'M&A', 'Treasury bonds']
Soda_vol   | OLS R2=1.0000 | selected=['Bank loans', 'Control stakes', 'Germany', 'Japan', 'M&A']
Beer_vol   | OLS R2=1.0000 | selected=['Bond yields', 'Currencies/metals', 'Electronics', 'Federal Reserve', 'Japan']
Smoke_vol  | OLS R2=1.0000 | selected=['Bush/Obama/Trump', 'Commodities', 'Japan', 'Reagan', 'Research']
Toys_

,outcome,alpha,selected,ols_r2,n_obs
0,mret,0.000004,California; Commodities; Currencies/metals; M&...,1.000000,4
25,ElcEq_vol,0.000011,C-suite; Commodities; Economic growth; Exchang...,1.000000,4
27,Aero_vol,0.000194,Bush/Obama/Trump; Environment ; Reagan; Resear...,1.000000,4
28,Ships_vol,0.037605,Accounting; Bank loans; California; News confe...,1.000000,4
29,Guns_vol,0.000425,Control stakes; Germany; Japan; M&A; Treasury ...,1.000000,4
30,Gold_vol,0.082487,Accounting; Bank loans; California; News confe...,1.000000,4
31,Mines_vol,0.000354,Bond yields; Broadcasting; Currencies/metals; ...,1.000000,4
32,Coal_vol,0.001551,Accounting; Bankruptcy; Economic growth; Excha...,1.000000,4
33,Oil_vol,0.000489,Bank loans; Commodities; Control stakes; Germa...,1.000000,4
35,Telcm_vol,0.002579,Control stakes; Foods/consumer goods; Germany;...,1.000000,4


---
## Q2(b) — Prompt engineering

### (i) Persona: bull vs bear

---


In [62]:
personas = [None, 'bull', 'bear']
persona_rows = []

for p in personas:
    scores = score_headlines(topics_union, temperature=BASE_TEMP, persona=p, ignore_future=False, limit=ARTICLE_LIMIT)
    monthly = monthly_aggregate(scores, topics_union)
    dfp = macro.reset_index().merge(monthly, on='date', how='inner').sort_values('date').reset_index(drop=True)

    for TARGET in outcomes:
        y = dfp.set_index('date')[TARGET].dropna()
        X = dfp.set_index('date').loc[y.index, topics_union]
        X_scaled = StandardScaler().fit_transform(X)

        alpha_5, *_ = lasso_pick_alpha_k_nonzero(X_scaled, y.values, k=5)
        lasso_5 = Lasso(alpha=alpha_5, max_iter=50_000).fit(X_scaled, y.values)
        selected = X.columns[lasso_5.coef_ != 0].tolist()
        r2, _ = ols_r2(X[selected].values, y.values)

        persona_rows.append({'persona': p or 'baseline', 'outcome': TARGET, 'ols_r2': r2})

persona_summary = pd.DataFrame(persona_rows)
persona_summary.to_csv(CACHE_DIR / 'q2b_persona_summary.csv', index=False)
persona_summary.pivot_table(index='outcome', columns='persona', values='ols_r2')


  [cache hit] llm_scores_model=gpt-4.1-nano__temp=0p3__persona=none__ignorefuture=0__limit=100.parquet
  [cache hit] llm_scores_model=gpt-4.1-nano__temp=0p3__persona=bull__ignorefuture=0__limit=100.parquet
  [cache hit] llm_scores_model=gpt-4.1-nano__temp=0p3__persona=bear__ignorefuture=0__limit=100.parquet


persona,baseline,bear,bull
outcome,,,
Aero_vol,1.000000,1.000000,1.000000
Agric_vol,1.000000,1.000000,1.000000
Autos_vol,1.000000,1.000000,1.000000
Banks_vol,1.000000,1.000000,1.000000
Beer_vol,1.000000,1.000000,1.000000
BldMt_vol,1.000000,1.000000,1.000000
Books_vol,1.000000,1.000000,1.000000
Boxes_vol,0.999919,1.000000,1.000000
BusSv_vol,1.000000,1.000000,1.000000


---
### (ii) Temperature: performance + stability

Stability metric: average pairwise Jaccard similarity of the binary topic vectors across repeated regenerations on the same sample.

---


In [63]:
temps = [0.0, 0.3, 0.7]
temp_rows = []

for t in temps:
    scores = score_headlines(topics_union, temperature=t, persona=None, ignore_future=False, limit=ARTICLE_LIMIT)
    monthly = monthly_aggregate(scores, topics_union)
    dft = macro.reset_index().merge(monthly, on='date', how='inner').sort_values('date').reset_index(drop=True)

    for TARGET in ['mret','vol','indpro','indprol1']:
        if TARGET not in outcomes:
            continue
        y = dft.set_index('date')[TARGET].dropna()
        X = dft.set_index('date').loc[y.index, topics_union]
        X_scaled = StandardScaler().fit_transform(X)

        alpha_5, *_ = lasso_pick_alpha_k_nonzero(X_scaled, y.values, k=5)
        lasso_5 = Lasso(alpha=alpha_5, max_iter=50_000).fit(X_scaled, y.values)
        selected = X.columns[lasso_5.coef_ != 0].tolist()
        r2, _ = ols_r2(X[selected].values, y.values)

        temp_rows.append({'temperature': t, 'outcome': TARGET, 'ols_r2': r2})

temp_summary = pd.DataFrame(temp_rows)
temp_summary.to_csv(CACHE_DIR / 'q2b_temperature_performance.csv', index=False)
temp_summary.pivot_table(index='outcome', columns='temperature', values='ols_r2')


  [cache hit] llm_scores_model=gpt-4.1-nano__temp=0p0__persona=none__ignorefuture=0__limit=100.parquet
  [cache hit] llm_scores_model=gpt-4.1-nano__temp=0p3__persona=none__ignorefuture=0__limit=100.parquet
  [cache hit] llm_scores_model=gpt-4.1-nano__temp=0p7__persona=none__ignorefuture=0__limit=100.parquet


temperature,0.0,0.3,0.7
outcome,,,
indpro,1.0,1.0,1.0
indprol1,1.0,1.0,1.0
mret,1.0,1.0,1.0
vol,1.0,1.0,1.0


In [64]:
# ── Temperature stability: re-score a small sample multiple times ──
def jaccard_vec(a, b):
    A, B = set(np.where(a > 0)[0]), set(np.where(b > 0)[0])
    if not A and not B: return 1.0
    if not A or not B: return 0.0
    return len(A & B) / len(A | B)

STAB_SAMPLE_N = 50
STAB_REPEATS = 3

sample_hls = articles['headline'].head(STAB_SAMPLE_N).tolist()

stab_rows = []
for t in temps:
    reps = []
    for r in range(STAB_REPEATS):
        # Parallel scoring (same ThreadPoolExecutor pattern as score_headlines)
        results = [None] * len(sample_hls)
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futs = {pool.submit(score_single_headline, hl, topics_union, t, None, False): i
                    for i, hl in enumerate(sample_hls)}
            for fut in as_completed(futs):
                idx = futs[fut]
                results[idx] = fut.result()
        reps.append(pd.DataFrame(results)[topics_union].to_numpy(int))
        print(f"  temp={t}, rep {r+1}/{STAB_REPEATS} done")

    sims = []
    for i in range(STAB_REPEATS):
        for j in range(i+1, STAB_REPEATS):
            n_rows = min(len(reps[i]), len(reps[j]))
            sims.extend([jaccard_vec(reps[i][k], reps[j][k]) for k in range(n_rows)])
    stab_rows.append({'temperature': t, 'avg_pairwise_jaccard': float(np.mean(sims)) if sims else np.nan})

stab = pd.DataFrame(stab_rows)
stab.to_csv(CACHE_DIR / 'q2b_temperature_stability.csv', index=False)
stab

  temp=0.0, rep 1/3 done
  temp=0.0, rep 2/3 done
  temp=0.0, rep 3/3 done
  temp=0.3, rep 1/3 done
  temp=0.3, rep 2/3 done
  temp=0.3, rep 3/3 done
  temp=0.7, rep 1/3 done
  temp=0.7, rep 2/3 done
  temp=0.7, rep 3/3 done


,temperature,avg_pairwise_jaccard
0,0.0,0.857778
1,0.3,0.799111
2,0.7,0.646556


---
### (iii) Lookahead bias (GFC sample)
We compare topic activations on 2007–2009 headlines with/without an explicit instruction to ignore future knowledge.

---


In [66]:
# ── Lookahead bias: score GFC headlines with and without ignore_future ──
gfc = articles[(articles['display_date'] >= '2007-01-01') & (articles['display_date'] <= '2009-12-31')].copy()
print('GFC-era headlines in full dataset:', len(gfc))

GFC_SAMPLE = min(200, len(gfc))

if GFC_SAMPLE > 0:
    gfc_s = gfc.sample(n=GFC_SAMPLE, random_state=0).reset_index(drop=True)
    hls = gfc_s['headline'].tolist()

    # Score this GFC sample directly (parallel, no caching — small batch)
    def _score_batch(headlines, temperature, persona, ignore_future):
        results = [None] * len(headlines)
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
            futs = {pool.submit(score_single_headline, hl, topics_union, temperature, persona, ignore_future): i
                    for i, hl in enumerate(headlines)}
            for fut in as_completed(futs):
                results[futs[fut]] = fut.result()
        return pd.DataFrame(results)

    print(f"Scoring {GFC_SAMPLE} GFC headlines (normal)...")
    scores_norm = _score_batch(hls, BASE_TEMP, None, False)
    print(f"Scoring {GFC_SAMPLE} GFC headlines (ignore future)...")
    scores_ign  = _score_batch(hls, BASE_TEMP, None, True)

    def active_topics(row):
        return ', '.join([t for t in topics_union if int(row.get(t, 0)) == 1][:20])

    out = pd.DataFrame({
        'display_date': gfc_s['display_date'].values,
        'headline': gfc_s['headline'].values,
        'active_topics_normal': scores_norm.apply(active_topics, axis=1).values,
        'active_topics_ignore_future': scores_ign.apply(active_topics, axis=1).values,
    })
    out.to_csv(CACHE_DIR / 'q2b_gfc_lookahead_samples.csv', index=False)
    display(out.head(10))
else:
    print('No 2007–2009 headlines found; skipping.')

GFC-era headlines in full dataset: 899
Scoring 200 GFC headlines (normal)...
Scoring 200 GFC headlines (ignore future)...


,display_date,headline,active_topics_normal,active_topics_ignore_future
0,2008-08-27 06:02:22.155,International Finance: Letter From The City / ...,International exchanges,International exchanges
1,2007-06-29 06:09:03.678,Heard on the Street: Discover Finds a Fan Base...,,Publishing
2,2008-05-12 06:11:33.344,THE OUTLOOK: Four Ways to Ease a Global Food C...,Environment,"Challenges, Changes, Environment , Problems"
3,2007-02-13 07:18:31.133,Corporate Focus: Tenaris to Pay $2.1 Billion T...,"C-suite, M&A","C-suite, M&A"
4,2008-11-04 07:13:22.483,World News: Toyota's Global Woes Start to Hit ...,"Automotive, Economic growth, Japan, Problems","Automotive, Challenges, Economic growth, Japan..."
5,2008-12-19 07:17:33.481,U.S. News: White House Says Help For Auto Make...,Automotive,Automotive
6,2009-01-22 07:10:32.628,World News: Israel Completes Gaza Pullout and ...,,
7,2008-04-04 06:16:12.469,Corporate News: GM Pushes Testing on Electric ...,Automotive,Automotive
8,2008-05-16 06:22:52.303,International Finance: Barclays Doesn't Budge ...,"C-suite, Earnings losses, Investment banking","C-suite, Earnings losses, Financial reports, I..."
9,2007-11-06 07:11:12.691,Business Technology: Big Computers Big Changes...,"Challenges, Changes, Microchips, Problems, Sys...","Challenges, Changes, Internet, Microchips, Pro..."


---
## Q2(c) — Forecasting indpro growth (indprol1)
We mirror HW3p1’s **expanding-window LassoCV** approach using the generated monthly topic features.

---


In [72]:
# ── Expanding-window forecast with LassoCV (mirrors HW3p1) ──
TARGET = 'indprol1'

y_series = df_base.set_index('date')[TARGET].dropna()
X_df = df_base.set_index('date').loc[y_series.index, topics_union]

y = y_series.values
X = X_df.values
dates = y_series.index

n = len(y)
MIN_TRAIN = max(2, n // 2)  # use half the data as initial training window

preds = np.full(n, np.nan)
scaler = StandardScaler().fit(X)
X_scaled = scaler.transform(X)

for t in range(MIN_TRAIN, n):
    n_splits = min(5, t - 1)  # CV folds must be < training size
    if n_splits < 2:
        # too few points for CV — use plain Lasso with a fixed alpha
        from sklearn.linear_model import Lasso as _L
        model = _L(alpha=0.01, max_iter=20_000).fit(X_scaled[:t], y[:t])
    else:
        tscv = TimeSeriesSplit(n_splits=n_splits)
        model = LassoCV(cv=tscv, alphas=50, max_iter=20_000)
        model.fit(X_scaled[:t], y[:t])
    preds[t] = model.predict(X_scaled[[t]])[0]

preds_oos = preds[MIN_TRAIN:]
actuals_oos = y[MIN_TRAIN:]
dates_oos = dates[MIN_TRAIN:]

# OOS R² vs expanding mean
expanding_mean = np.array([y[:t].mean() for t in range(MIN_TRAIN, n)])
ss_res = np.sum((actuals_oos - preds_oos) ** 2)
ss_tot = np.sum((actuals_oos - expanding_mean) ** 2)
oos_r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

print(f"n={n} months, MIN_TRAIN={MIN_TRAIN}, OOS obs={len(preds_oos)}")
print(f"OOS R² (generated topics) = {oos_r2:.4f}")
if len(dates_oos) > 0:
    print(f"Evaluation: {dates_oos[0].date()} → {dates_oos[-1].date()} ({len(dates_oos)} months)")
if n < 50:
    print(f"⚠ Only {n} months — increase ARTICLE_LIMIT for meaningful results.")

n=4 months, MIN_TRAIN=2, OOS obs=2
OOS R² (generated topics) = 0.0000
Evaluation: 1984-03-01 → 1984-04-01 (2 months)
⚠ Only 4 months — increase ARTICLE_LIMIT for meaningful results.


In [73]:
# Save forecast table
forecast_tbl = pd.DataFrame({'date': dates_oos, 'y_true': actuals_oos, 'y_pred': preds_oos})
forecast_tbl.to_csv(CACHE_DIR / 'q2c_forecast_indprol1_baseline.csv', index=False)

forecast_tbl.head()


,date,y_true,y_pred
0,1984-03-01,0.005914,0.004665
1,1984-04-01,0.005345,0.005081


---
### Notes for writeup
- Compare Q2(a) R² to your Q1 topic-based R².
- Comment on persona and temperature changes.
- Use the GFC sample table to discuss lookahead bias.
- Compare Q2(c) forecasting OOS R² to Q1(c).
